## Transform Refunds Data
1. Extract specific portion of the string from refund_reason using split function
1. Extract specific portion of the string from refund_reason using regexp_extract function
1. Extract date and time from the refund_timestamp
1. Write transformed data to the Silver schema

In [0]:
df_refunds = spark.table('gizmobox.bronze.py_refunds')
display(df_refunds)

### 1. Extract specific portion of the string from refund_reason using split function

In [0]:
from pyspark.sql import functions as F
df_split_refunds = (
    df_refunds
        .select(
            "refund_id",
            "payment_id",
            "refund_timestamp",
            "refund_amount",
            F.split("refund_reason", ":")[0].alias("refund_reason"),
            F.split("refund_reason", ":")[1].alias("refund_source")
        )
)
display(df_split_refunds)

### 2. Extract specific portion of the string from refund_reason using regexp_extract function

In [0]:
df_transformed_refunds = (
    df_refunds
        .select(
            "refund_id",
            "payment_id",
            F.date_format('refund_timestamp','yyyy-MM-dd').cast('date').alias('refund_date'),
            F.date_format('refund_timestamp','HH-mm-ss').alias('refund_time'),
            "refund_amount",
            F.regexp_extract("refund_reason", "^([^:]+):", 1).alias("refund_reason"),
            F.regexp_extract("refund_reason", "^[^:]+:(.*)$", 1).alias("refund_source")
        )
)
display(df_transformed_refunds)

### 3. Write transformed data to the Silver schema  

In [0]:
df_transformed_refunds.writeTo("gizmobox.silver.py_refunds").createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_refunds;